# Lab 3 — Build and serve the customized model with NVIDIA Riva

Lab 2 produced a complete NeMo `.nemo` checkpoint. Here, NVIDIA `nemo2riva` exports it to `.riva`; the matching Parakeet ASR NIM image then provides Riva ServiceMaker to build a hardware-independent RMIR and deploy an optimized model repository. Riva uses TensorRT and NVIDIA Triton internally, while applications call the supported Riva gRPC API.

Two deployment paths are provided:

1. **Amazon EKS (production path):** deploy the custom RMIR from S3 with the Speech NIM Helm chart on a GPU-enabled EKS cluster.
2. **Local Docker on Brev (workshop path):** build, optimize, serve, and call the same Riva pipeline on this single GPU.

This notebook uses the **Own Your Voice Riva Client** kernel. The setup script installs it separately because NeMo 2.7.3 and Riva client 2.26.0 require incompatible Protobuf versions. Jupyter should select it from the notebook metadata; choose it manually if prompted.


In [ ]:
from getpass import getpass
from pathlib import Path
import json, os, subprocess, sys, time, wave

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'voice_asr_lab').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'voice_asr_lab').exists():
    raise RuntimeError('Open this notebook from the workshop repository.')
sys.path.insert(0, str(ROOT / 'src'))

import riva.client
from jiwer import wer
from voice_asr_lab.audio import normalize_latin_text
from voice_asr_lab.nemo import read_nemo_manifest


## 1. Confirm the artifact and deployment controls

The published Parakeet 0.6B ASR NIM 3.1.0 supplies `riva-build`, `riva-deploy`, and the server. The isolated Riva Python client remains at 2.26.0; its package version does not select the server image. The deploy phase must run on the target GPU because TensorRT engines are GPU-specific. The RMIR itself can be transferred to EKS. Set `SAVE_INTERMEDIATE_ONNX = True` below if you also want a standalone ONNX copy for inspection or a follow-on exercise. This adds a second export and is not required by Riva. The build requires 20 GB free by default and stops before export when the checked artifact filesystem is too full.


In [ ]:
ASR_NIM_CONTAINER_ID = 'parakeet-0-6b-ctc-en-us'
ASR_NIM_TAG = '3.1.0'
NEMO_MODEL = ROOT / 'artifacts' / 'parakeet-ctc-0.6b-nl.nemo'
SAVE_INTERMEDIATE_ONNX = False
RIVA_MIN_FREE_GB = 20
ONNX_FILE = ROOT / 'artifacts' / 'onnx' / 'parakeet-ctc-0.6b-nl.onnx'
RMIR_FILE = ROOT / 'artifacts' / 'riva' / 'own_your_voice_asr.rmir'
PIPELINE_NAME = 'own-your-voice-nl-asr-offline'
RIVA_URI = '127.0.0.1:50051'
LANGUAGE_CODE = 'nl-NL'

if not NEMO_MODEL.is_file():
    raise RuntimeError(f'Missing {NEMO_MODEL}. Complete Lab 2 first.')
subprocess.run(['bash', str(ROOT / 'scripts' / 'stop_nim.sh')], check=True)
print({
    'nemo_model': str(NEMO_MODEL),
    'nemo_size_gb': round(NEMO_MODEL.stat().st_size / 1024**3, 2),
    'asr_nim_image': f'nvcr.io/nim/nvidia/{ASR_NIM_CONTAINER_ID}:{ASR_NIM_TAG}',
    'save_intermediate_onnx': SAVE_INTERMEDIATE_ONNX,
    'minimum_free_disk_gb': RIVA_MIN_FREE_GB,
    'riva_client_version': '2.26.0', 'pipeline': PIPELINE_NAME,
})


## 2. Build the Riva Model Intermediate Representation

The NVIDIA reference notebook uses standalone `nemo2riva`, and the ASR NIM 3.1.0 image exposes the positional ServiceMaker CLI. The helper therefore exports `.nemo → .riva` with the supported Parakeet-CTC settings, ONNX opset 19 and `max_dim=1000`, then calls `riva-build speech_recognition` to produce the RMIR. When the optional ONNX toggle is enabled, the helper first asks NeMo 2.7.3 to save `artifacts/onnx/parakeet-ctc-0.6b-nl.onnx` from the same checkpoint. That standalone file is retained for inspection; `nemo2riva` still performs the packaged conversion used by Riva. The NGC key is hidden and supplied only to the subprocess environment. With this pinned converter, the `Unsupported model IR version: 11, max supported IR version: 10` inference-validation warning is non-fatal if conversion continues; `No space left on device` is a separate fatal error.


In [ ]:
ngc_api_key = getpass('NGC API key (input is hidden): ').strip()
if not ngc_api_key:
    raise ValueError('An NGC API key is required for the ASR NIM container.')
riva_env = {
    **os.environ, 'NGC_API_KEY': ngc_api_key,
    'ASR_NIM_CONTAINER_ID': ASR_NIM_CONTAINER_ID, 'ASR_NIM_TAG': ASR_NIM_TAG,
    'NEMO_MODEL': str(NEMO_MODEL), 'RIVA_PIPELINE_NAME': PIPELINE_NAME,
    'SAVE_INTERMEDIATE_ONNX': '1' if SAVE_INTERMEDIATE_ONNX else '0',
    'RIVA_MIN_FREE_GB': str(RIVA_MIN_FREE_GB),
    'ONNX_MODEL': str(ONNX_FILE),
}
subprocess.run(
    ['bash', str(ROOT / 'scripts' / 'build_riva_rmir.sh')],
    check=True, env=riva_env,
)
if not RMIR_FILE.is_file() or RMIR_FILE.stat().st_size == 0:
    raise RuntimeError('riva-build completed without a usable RMIR.')
build_artifacts = {
    'rmir': str(RMIR_FILE),
    'rmir_size_gb': round(RMIR_FILE.stat().st_size / 1024**3, 2),
}
if SAVE_INTERMEDIATE_ONNX:
    if not ONNX_FILE.is_file() or ONNX_FILE.stat().st_size == 0:
        raise RuntimeError('ONNX export was enabled but no usable file was created.')
    build_artifacts.update({
        'onnx': str(ONNX_FILE),
        'onnx_size_mb': round(ONNX_FILE.stat().st_size / 1024**2, 2),
    })
print(build_artifacts)


## 3A. Production path — deploy the RMIR on Amazon EKS

This path assumes a pre-existing GPU-enabled EKS cluster; it is an instructor or platform-team exercise because creating AWS infrastructure is outside the attendee notebook. The Speech NIM Helm chart loads the custom RMIR from S3, runs GPU-specific optimization, creates the Triton repository, starts the Riva APIs, and exposes a Kubernetes Service.


In [ ]:
eks_guide = ROOT / 'deploy' / 'eks' / 'README.md'
values_override = ROOT / 'deploy' / 'eks' / 'values-custom-rmir.yaml'
print(eks_guide.read_text(encoding='utf-8'))
print('--- Helm model override ---')
print(values_override.read_text(encoding='utf-8'))


### EKS execution gate

The next cell performs read-only checks only when enabled. It never creates a cluster or installs Helm automatically. Use the guide after an AWS/EKS owner confirms the cluster, Region, GPU node capacity, storage class, NGC entitlement, and ingress policy.


In [ ]:
CHECK_EKS_PREREQUISITES = False
if CHECK_EKS_PREREQUISITES:
    for command in (
        ['aws', 'sts', 'get-caller-identity'],
        ['kubectl', 'cluster-info'],
        ['kubectl', 'get', 'nodes', '-o', 'wide'],
    ):
        subprocess.run(command, check=True)
else:
    print('EKS checks skipped. Use deploy/eks/README.md with the platform owner.')


## 3B. Workshop path — deploy the custom Riva ASR NIM on the Brev GPU

This executes `riva-deploy` on the current GPU, packages the generated repository as `custom_model.tar.gz`, and serves it with the ASR NIM. First deployment can take many minutes. HTTP health checks use port 9000 and applications use Riva gRPC on port 50051; Triton ports remain internal.


In [ ]:
try:
    subprocess.run(
        ['bash', str(ROOT / 'scripts' / 'start_riva.sh')],
        check=True, env=riva_env,
    )
finally:
    del riva_env['NGC_API_KEY']
    del ngc_api_key


## 4. Call the Riva gRPC API

We send raw 16-bit PCM from a held-out Dutch WAV and explicitly select the deployed pipeline. The same client works through `kubectl port-forward` for the EKS path.


In [ ]:
test_manifest = ROOT / 'artifacts' / 'nemo_manifests' / 'nl_test.jsonl'
test_rows = read_nemo_manifest(test_manifest)
sample = test_rows[0]
with wave.open(sample['audio_filepath'], 'rb') as wav_file:
    frame_count = wav_file.getnframes()
    sample_rate = wav_file.getframerate()
    channels = wav_file.getnchannels()
    sample_width = wav_file.getsampwidth()
    audio_bytes = wav_file.readframes(frame_count)

audio_check = {
    'path': sample['audio_filepath'], 'frames': frame_count,
    'sample_rate': sample_rate, 'channels': channels,
    'sample_width_bytes': sample_width, 'audio_bytes': len(audio_bytes),
    'duration_seconds': frame_count / sample_rate if sample_rate else 0,
}
print({'audio_check': audio_check})
if not audio_bytes or frame_count <= 0:
    raise RuntimeError('Selected WAV contains no audio frames.')
if sample_rate != 16_000 or channels != 1 or sample_width != 2:
    raise RuntimeError('Riva example requires mono 16 kHz, 16-bit PCM WAV audio.')

auth = riva.client.Auth(uri=RIVA_URI)
asr_service = riva.client.ASRService(auth)
recognition_config = riva.client.RecognitionConfig(
    encoding=riva.client.AudioEncoding.LINEAR_PCM,
    sample_rate_hertz=sample_rate, audio_channel_count=channels,
    language_code=LANGUAGE_CODE, model=PIPELINE_NAME,
    max_alternatives=1, enable_automatic_punctuation=False,
)
response = asr_service.offline_recognize(audio_bytes, recognition_config)
if not response.results or not response.results[0].alternatives:
    raise RuntimeError('Riva returned no transcript alternatives.')
prediction = response.results[0].alternatives[0].transcript
reference = sample['text']
print({'reference': reference, 'prediction': prediction})


## 5. Measure service-boundary correctness and latency


In [ ]:
latencies = []
for _ in range(5):
    started = time.perf_counter()
    measured = asr_service.offline_recognize(audio_bytes, recognition_config)
    latencies.append(time.perf_counter() - started)
measured_prediction = measured.results[0].alternatives[0].transcript
audio_seconds = float(sample['duration'])
median_latency = sorted(latencies)[len(latencies) // 2]
service_report = {
    'sample_wer': wer(
        normalize_latin_text(reference), normalize_latin_text(measured_prediction)
    ),
    'audio_seconds': audio_seconds,
    'median_latency_seconds': median_latency,
    'real_time_factor': median_latency / audio_seconds,
    'throughput_x_realtime': audio_seconds / median_latency,
    'api': 'Riva gRPC', 'uri': RIVA_URI, 'model': PIPELINE_NAME,
}
report_path = ROOT / 'artifacts' / 'lab3_riva_report.json'
report_path.write_text(json.dumps(service_report, indent=2) + '\n', encoding='utf-8')
service_report


## Production handoff and cleanup

The local result proves the NeMo → Riva archive → RMIR → custom ASR NIM → Riva API artifact chain on one GPU. It is not an EKS scale result. For production, optimize on the exact target GPU, keep nodes homogeneous when reusing model caches, use the chart's S3 RMIR support and persistent model cache, configure an HTTP/2/gRPC ingress with TLS, add health checks and telemetry, and load-test latency, throughput, concurrency, GPU memory, and held-out WER.

Stop the local service when finished: `bash scripts/stop_riva.sh`.
